# K-Means + UMAP Lab

Pipeline minh hoạ: chuẩn hoá dữ liệu → tìm số cụm tối ưu → fit K-Means → giảm chiều bằng UMAP để trực quan hoá.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from umap import UMAP

## 1. Load & Scale

In [ ]:
wine = datasets.load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled[:3]

## 2. Elbow & Silhouette

In [ ]:
k_range = range(2, 11)
inertias, silhouettes = [], []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(k_range, inertias, marker='o')
ax[0].set_title('Elbow (Inertia)')
ax[0].set_xlabel('k')
ax[0].set_ylabel('Inertia')
ax[1].plot(k_range, silhouettes, marker='o', color='orange')
ax[1].set_title('Silhouette Score')
ax[1].set_xlabel('k')
plt.show()

## 3. Fit K-Means & Project bằng UMAP

In [ ]:
best_k = 3
kmeans = KMeans(n_clusters=best_k, n_init=50, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)
umap = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embedding = umap.fit_transform(X_scaled)
plot_df = pd.DataFrame({
    'x': embedding[:, 0],
    'y': embedding[:, 1],
    'cluster': cluster_labels
})
sns.scatterplot(data=plot_df, x='x', y='y', hue='cluster', palette='tab10')
plt.title('UMAP projection with K-Means clusters')
plt.show()

## 4. Kết luận nhanh
- Elbow + silhouette giúp chọn `k`.
- UMAP cho visual rõ các cụm.
- Có thể thay dataset bằng embedding khách hàng/time-series.